## A notebook to get anatomical structures that express a given biomarker

## Install and import libraries

In [ ]:

%pip install pandas requests hra_jupyter_widgets ipywidgets matplotlib numpy

from pathlib import Path
import pandas as pd
import requests
from enum import Enum, IntEnum, StrEnum, Flag, auto
from hra_jupyter_widgets import (BodyUi) #for visualization in 3D
from pprint import pprint
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

## Define variables

In [ ]:
class Biomarker_Identifier_Type(Enum):
    GENE_ID = "gene_id"
    GENE_LABEL = "gene_label"
    ENSEMBL_ID = "ensembl_id"

## Load data

In [ ]:
hra_pop_version = "v1.1"
branch = "main"
data_dir = Path("data/")
file_name_data = "datasets-as-ct-gene.csv.gz"
file_path_data = data_dir / file_name_data
data_url = f"https://github.com/x-atlas-consortia/hra-pop/raw/refs/heads/{branch}/output-data/{hra_pop_version}/reports/atlas-ad-hoc/datasets-as-ct-gene.csv.gz"

In [ ]:
file_name_3d = "rui-reference-data.json"
file_path_3d = data_dir / file_name_3d
as_3d_url = "https://apps.humanatlas.io/api/v1/rui-reference-data"

In [ ]:
def download_or_load_locally(url: str, file_path: Path, headers=None):

    if file_path.is_file():
        print(f"{file_path} already exists.")
    else:
        file_path.parent.mkdir(parents=True, exist_ok=True)
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        file_path.write_bytes(response.content)

In [ ]:
download_or_load_locally(data_url, file_path_data)

In [ ]:
download_or_load_locally(as_3d_url, file_path_3d, headers={"Accept": "application/json"})

with open(file_path_3d, encoding="utf-8") as f:
    rui_reference_data = json.load(f)

In [ ]:
df_data = pd.read_csv(file_path_data)

In [ ]:
df_data.head()

## Functions

In [ ]:
def filter_by_biomarker(id:str, biomarker_id_type:Biomarker_Identifier_Type) -> pd.DataFrame:
    match biomarker_id_type:
        case Biomarker_Identifier_Type.GENE_ID:
            df_result = df_data[df_data[Biomarker_Identifier_Type.GENE_ID.value] == id]
        case Biomarker_Identifier_Type.GENE_LABEL:
            df_result = df_data[df_data[Biomarker_Identifier_Type.GENE_LABEL.value] == id]
        case Biomarker_Identifier_Type.ENSEMBL_ID:
            df_result = df_data[df_data[Biomarker_Identifier_Type.ENSEMBL_ID.value] == id]
    return df_result

def curie_to_uri(curie:str):
    if curie.startswith("http"):
        return curie
    elif "FMA" in curie:
        return "http://purl.org/sig/ont/fma/fma" + curie.split(":")[-1]
    else:
        return "http://purl.obolibrary.org/obo/UBERON_" + curie.split(":")[-1]

def map_mean_expr_to_color(mean_expr: float, vmin: float, vmax: float) -> list[int]:
  # Normalize mean_expr to [0, 1] relative to the range of all values
  norm = plt.Normalize(vmin=vmin, vmax=vmax, clip=True)

  # Look up the viridis color (RGBA floats in [0, 1])
  r, g, b, _ = plt.colormaps["viridis"](norm(mean_expr))

  # Convert to 0-255 integers
  return [round(r * 255), round(g * 255), round(b * 255)]

In [ ]:
result = df_data.groupby("gene_id")["mean_expr"].agg(lambda x: x.max() - x.min()).idxmax()
result

In [ ]:
df_data[df_data["gene_id"] == "MT-RNR2"]

## Get user-provided biomarker expression in AS

In [ ]:
# Replace the placeholder below with your biomarker of interest
MY_GENE_LABEL = "MT-RNR2"
MY_GENE_SYMBOL = "MT-RNR2"
MY_ENSEMBL_ID = "ENSG00000210082.2"

In [ ]:
df_result_b = filter_by_biomarker(MY_GENE_LABEL, Biomarker_Identifier_Type.GENE_LABEL)
df_result_b.head()

In [ ]:
df_result_b = filter_by_biomarker(MY_GENE_SYMBOL, Biomarker_Identifier_Type.GENE_ID)
df_result_b.head()

In [ ]:
df_result_b = filter_by_biomarker(MY_ENSEMBL_ID, Biomarker_Identifier_Type.ENSEMBL_ID)
df_result_b.head()

## Provide AS, get Bs

In [ ]:
# Replace the placeholder below with your AS of interest
MY_AS_ID = "UBERON:0002080"

df_result_as = df_data[df_data["as_id"] == MY_AS_ID]
df_result_as.head()

## ProvideCT, get Bs

In [ ]:
# Replace the placeholder below with your CT of interest
MY_CT_ID = "CL:0000097"

df_result_ct = df_data[df_data["cl_id"] == MY_CT_ID]
df_result_ct.head()

## Visualize with HRA Jupyter widget

In [ ]:
df_b_mean = df_result_b.groupby("as_id")["mean_expr"].mean().reset_index()
df_b_mean["as_id_uri"] = df_b_mean["as_id"].apply(lambda id: curie_to_uri(id))

df_b_mean

In [ ]:
with open("data/rui-reference-data.json", "r") as f:
    rui_reference_data = json.load(f)

# Get relevant nodes in the kidney
sceneNodesSelected = []
for organ_iri, value in rui_reference_data["sceneNodeLookup"].items():
    if (
        value.get("representation_of") in df_b_mean["as_id_uri"].tolist()
        and value.get("reference_organ")
        == "https://purl.humanatlas.io/ref-organ/large-intestine-female/v1.3#primary"
    ):
        # print("yes")
        sceneNodesSelected.append(value)

In [ ]:
# Assign colors by B expression, normalized over the displayed nodes only
expr_lookup = dict(zip(df_b_mean["as_id_uri"], df_b_mean["mean_expr"]))
selected_expr = [expr_lookup[node["representation_of"]] for node in sceneNodesSelected]
vmin, vmax = min(selected_expr), max(selected_expr)

for node in sceneNodesSelected:
  node["_lighting"] = "flat"
  rgb = map_mean_expr_to_color(expr_lookup[node["representation_of"]], vmin, vmax)
  node["color"] = rgb + [255]

In [ ]:
body_ui = BodyUi(scene=sceneNodesSelected)
display(body_ui)